# 01 — Data Ingestion
Load, document, fix timestamps, map sentiment labels, merge on date, save.

**Input:** `data/raw/fear_greed_index.csv`, `data/raw/historical_data.csv`
**Output:** `data/processed/merged_data.csv`

In [10]:
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os
from src.config import SENTIMENT_FILE, TRADER_FILE, MERGED_FILE, DATA_PROCESSED, SENTIMENT_MAP
from src.utils import describe_df, print_section
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load raw datasets

In [11]:
sentiment_df = pd.read_csv(SENTIMENT_FILE)
trader_df    = pd.read_csv(TRADER_FILE)
print(f'Sentiment : {sentiment_df.shape[0]:,} rows x {sentiment_df.shape[1]} columns')
print(f'Trader    : {trader_df.shape[0]:,} rows x {trader_df.shape[1]} columns')

Sentiment : 2,644 rows x 4 columns
Trader    : 211,224 rows x 16 columns


## 2. Document sentiment dataset

In [12]:
describe_df(sentiment_df, 'SENTIMENT DATASET')
print()
print('Raw classification values:')
print(sentiment_df['classification'].value_counts())
sentiment_df.head()


  SENTIMENT DATASET
  Shape      : 2,644 rows x 4 columns
  Columns    : ['timestamp', 'value', 'classification', 'date']
  Dtypes     :
             timestamp                      int64
             value                          int64
             classification                 object
             date                           object
  Missing    :
             timestamp                      0
             value                          0
             classification                 0
             date                           0
  Duplicates : 0

Raw classification values:
classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


## 3. Document trader dataset

In [13]:
describe_df(trader_df, 'TRADER DATASET')
print()
print(f'Unique accounts : {trader_df["Account"].nunique():,}')
print(f'Unique coins    : {trader_df["Coin"].nunique()}')
print(f'Side values     : {trader_df["Side"].unique()}')
print(f'Direction values: {trader_df["Direction"].unique()}')
trader_df.head()


  TRADER DATASET
  Shape      : 211,224 rows x 16 columns
  Columns    : ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
  Dtypes     :
             Account                        object
             Coin                           object
             Execution Price                float64
             Size Tokens                    float64
             Size USD                       float64
             Side                           object
             Timestamp IST                  object
             Start Position                 float64
             Direction                      object
             Closed PnL                     float64
             Transaction Hash               object
             Order ID                       int64
             Crossed                        bool
             Fee                   

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.8700,7872.1600,BUY,02-12-2024 22:50,0.0000,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.3454,895000000000000.0000,1730000000000.0000
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.0000,127.6800,BUY,02-12-2024 22:50,986.5246,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0056,443000000000000.0000,1730000000000.0000
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.0900,1150.6300,BUY,02-12-2024 22:50,1002.5190,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0504,660000000000000.0000,1730000000000.0000
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.9800,1142.0400,BUY,02-12-2024 22:50,1146.5586,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0500,1080000000000000.0000,1730000000000.0000
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.7300,69.7500,BUY,02-12-2024 22:50,1289.4885,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0031,1050000000000000.0000,1730000000000.0000


## 4. Map 5 sentiment labels to 2 (Fear / Greed)

In [14]:
sentiment_df['Classification'] = sentiment_df['classification'].map(SENTIMENT_MAP)
print('After mapping:')
print(sentiment_df['Classification'].value_counts())
print(f'Unmapped (NaN): {sentiment_df["Classification"].isnull().sum()}')

After mapping:
Classification
Fear     1685
Greed     959
Name: count, dtype: int64
Unmapped (NaN): 0


## 5. Fix dates and timestamps

In [15]:
sentiment_df['date'] = pd.to_datetime(sentiment_df['date']).dt.date

ts_sample = trader_df['Timestamp'].iloc[0]
print(f'Timestamp sample: {ts_sample}')

if ts_sample > 1e12:
    trader_df['date'] = pd.to_datetime(trader_df['Timestamp'], unit='ms').dt.date
else:
    trader_df['date'] = pd.to_datetime(trader_df['Timestamp'], unit='s').dt.date

print(f'Sentiment date range: {sentiment_df["date"].min()} to {sentiment_df["date"].max()}')
print(f'Trader date range   : {trader_df["date"].min()} to {trader_df["date"].max()}')

Timestamp sample: 1730000000000.0
Sentiment date range: 2018-02-01 to 2025-05-02
Trader date range   : 2023-03-28 to 2025-06-15


## 6. Check date overlap

In [16]:
s_min = pd.to_datetime(sentiment_df['date'].min())
s_max = pd.to_datetime(sentiment_df['date'].max())
t_min = pd.to_datetime(trader_df['date'].min())
t_max = pd.to_datetime(trader_df['date'].max())
overlap_start = max(s_min, t_min)
overlap_end   = min(s_max, t_max)
print(f'Sentiment range  : {s_min.date()} to {s_max.date()}')
print(f'Trader range     : {t_min.date()} to {t_max.date()}')
print(f'Overlap          : {overlap_start.date()} to {overlap_end.date()}')
print(f'Overlap days     : {(overlap_end - overlap_start).days}')

Sentiment range  : 2018-02-01 to 2025-05-02
Trader range     : 2023-03-28 to 2025-06-15
Overlap          : 2023-03-28 to 2025-05-02
Overlap days     : 766


## 7. Merge and save

In [17]:
merged_df = trader_df.merge(
    sentiment_df[['date', 'Classification']],
    on='date',
    how='inner'
)
print(f'Merged shape         : {merged_df.shape[0]:,} rows x {merged_df.shape[1]} columns')
print(f'Trades on Fear days  : {(merged_df["Classification"] == "Fear").sum():,}')
print(f'Trades on Greed days : {(merged_df["Classification"] == "Greed").sum():,}')
print(f'Unique accounts      : {merged_df["Account"].nunique():,}')
print(f'Unique days          : {merged_df["date"].nunique()}')
os.makedirs(DATA_PROCESSED, exist_ok=True)
merged_df.to_csv(MERGED_FILE, index=False)
print(f'Saved: {MERGED_FILE}')

Merged shape         : 184,263 rows x 18 columns
Trades on Fear days  : 141,012
Trades on Greed days : 43,251
Unique accounts      : 32
Unique days          : 6
Saved: /app/data/processed/merged_data.csv


In [18]:
merged_df.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,Classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.8700,7872.1600,BUY,02-12-2024 22:50,0.0000,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.3454,895000000000000.0000,1730000000000.0000,2024-10-27,Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.0000,127.6800,BUY,02-12-2024 22:50,986.5246,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0056,443000000000000.0000,1730000000000.0000,2024-10-27,Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.0900,1150.6300,BUY,02-12-2024 22:50,1002.5190,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0504,660000000000000.0000,1730000000000.0000,2024-10-27,Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.9800,1142.0400,BUY,02-12-2024 22:50,1146.5586,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0500,1080000000000000.0000,1730000000000.0000,2024-10-27,Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.7300,69.7500,BUY,02-12-2024 22:50,1289.4885,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.0031,1050000000000000.0000,1730000000000.0000,2024-10-27,Greed
